# 01 · Quickstart: a quantum classifier in four lines

**Goal:** fit Qlassifier on real data, read what it would cost on a quantum computer, and compare it with the standard quantum baseline.

What you need:
- Python 3.12 and `pip install .` from the repository root (see the [README](../README.md)).
- Nothing else: everything runs offline, on an exact simulation of the circuits.
- About a minute of compute.

Next: [02 · Geometry mode](02-geometry-mode.ipynb) · [03 · Reproduce](03-reproduce.ipynb) · [paper](../docs/paper/main.tex)

## 1. Fit and score

The Breast Cancer data (569 tumours, 30 measurements each) is bundled with the library.
`split(seed=0)` is the benchmark's own 70/30 stratified split.

In [1]:
from qlassifier import Qlassifier, load_dataset

X_train, X_test, y_train, y_test = load_dataset("breast_cancer").split(seed=0)
clf = Qlassifier().fit(X_train, y_train)
print(f"test accuracy: {clf.score(X_test, y_test):.3f}")

test accuracy: 0.947


What `fit` did, in order:
- **Standardise** each column on the training rows.
- **Encode** two columns per qubit, with fixed rotations: 30 columns → 15 qubits, no two-qubit gates.
- **Read out** each qubit's Bloch vector. A device would run 3 circuits per sample (X, Y and Z bases); the default backend computes them exactly.
- **Tune itself**: 5-fold cross-validation over 30 settings of kernel width (`gamma`) and margin (`C`), on the exact Bloch vectors. This costs zero circuits.
- **Train** a support vector machine on a Gaussian kernel of the Bloch vectors.

In [2]:
clf.resources_

Resources(backend='exact', qubits=15, two_qubit_gates=0, depth=5, circuits_per_sample=3, circuits_fit=1194, circuits_predict=513, tuning_circuits=0, shots=None)

In [3]:
print("qubits:", clf.n_qubits_)
print("chosen settings:", clf.best_params_)

qubits: 15
chosen settings: {'gamma': 1.0, 'C': 10.0}


`Qlassifier` is a scikit-learn estimator.
It works with `clone`, `Pipeline`, `GridSearchCV` and pickling, and it has `predict_proba`:

In [4]:
clf.predict_proba(X_test[:5]).round(3)

array([[1.   , 0.   ],
       [0.978, 0.022],
       [0.994, 0.006],
       [0.02 , 0.98 ],
       [0.975, 0.025]])

## 2. Against the standard quantum baseline

The challenge's standard quantum model is Qiskit's ZZ feature map with a quantum support vector classifier (ZZ QSVC).
`benchmark` runs models under one fair protocol: same splits, same 8 PCA features, same cross-validated tuning.
Two seeds keep this quick; the shipped results use ten.

In [5]:
from qlassifier import benchmark

small = benchmark("breast_cancer", seeds=2, models=["trivial", "zz_qsvc", "qlassifier"], n_jobs=2)
print(small.summary())

Breast Cancer (Wisconsin) · 3 models × 2 seeds · pca8 · BA (corrected 95% CI) · complete
model                        mean ± sd           95% CI       Qlassifier − model  Qlassifier wins
Qlassifier               0.934 ± 0.009   [0.826, 1.000]                        —                —
ZZ QSVC                  0.548 ± 0.014   [0.372, 0.724]  +38.7 pp [+32.0, +45.4]              2/2
Majority class           0.500 ± 0.000   [0.500, 0.500]  +43.4 pp [+32.6, +54.3]              2/2
ties need ≥ 5 seeds (none marked at 2)


How to read it:
- **Majority class** always predicts "benign": its balanced accuracy (BA) is 0.5 by definition.
- **ZZ QSVC** sits close to it: the standard quantum model barely learns this data.
- **Qlassifier** is far above both.
- With two seeds the confidence intervals are wide. Ten seeds narrow them.

## 3. The full leaderboard

The library ships the ten-seed result of every bundled dataset, with every baseline the challenge names.
`BenchmarkResult.cached` loads it instantly.

In [6]:
from qlassifier import BenchmarkResult

full = BenchmarkResult.cached("breast_cancer")
full.plot("leaderboard");

In [7]:
h = full.highlights
q = full.model("qlassifier").metrics.balanced_accuracy
best = full.model(h.best_classical)
d = h.vs_best_classical
print(f"Qlassifier BA {q.mean:.3f}, 95% CI [{q.ci_low:.3f}, {q.ci_high:.3f}]")
print(f"vs ZZ QSVC:    {100 * h.vs_standard_qml.mean_diff:+.1f} pp")
print(f"vs {best.name} (best classical): {100 * d.mean_diff:+.1f} pp, 95% CI [{100 * d.ci_low:+.1f}, {100 * d.ci_high:+.1f}]")

Qlassifier BA 0.948, 95% CI [0.928, 0.968]
vs ZZ QSVC:    +37.6 pp
vs MLP (best classical): -1.3 pp, 95% CI [-3.1, +0.5]


**Result.**
- Qlassifier beats the standard quantum baseline by a wide margin.
- Against the best tuned classical model, the confidence interval of the difference contains zero: a statistical tie, not a win.
- Qlassifier prepares product states, which classical computers simulate exactly. So this is a better quantum model, not a quantum advantage.

## 4. Same model, IBM device noise

`backend="fake_fez"` runs the final circuits on a simulator with the noise model of IBM's Heron r2 device (Qiskit Aer).
Tuning still happens on exact features, so the device runs only 3 circuits per training and test row.

In [8]:
noisy = Qlassifier(backend="fake_fez", random_state=0).fit(X_train, y_train)
print(f"test accuracy under the FakeFez noise model: {noisy.score(X_test, y_test):.3f}")
noisy.resources_

test accuracy under the FakeFez noise model: 0.959


Resources(backend='fake_fez', qubits=15, two_qubit_gates=0, depth=10, circuits_per_sample=3, circuits_fit=1194, circuits_predict=513, tuning_circuits=0, shots=1024)

**Assumptions to keep in mind.**
- The noise model is a simulation of an IBM device, not the device itself: no crosstalk, drift or leakage.
- `resources_` reports what the device would run: qubits, circuits, two-qubit gates and depth after transpilation.

Next: [02 · Geometry mode](02-geometry-mode.ipynb) shows what changes when the data are angles.